In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf

spark = (SparkSession.builder
         .appName("KafkaConsumer")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate()
)

spark

In [2]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:29092")
      .option("subscribe", "plane_data")
      .option("startingOffsets", "earliest")
      .load()
)

kafka_json_df = df.withColumn("value", sf.col("value").cast("string"))

In [3]:
from pyspark.sql.types import *
 
json_schema = StructType([
    StructField("time", LongType(), True),
    StructField("states", ArrayType(
        StructType([
            StructField("icao24", StringType(), True),
            StructField("callsign", StringType(), True),
            StructField("origin_country", StringType(), True),
            StructField("time_position", LongType(), True),
            StructField("last_contact", LongType(), True),
            StructField("longitude", DoubleType(), True),
            StructField("latitude", DoubleType(), True),
            StructField("baro_altitude", DoubleType(), True),
            StructField("on_ground", BooleanType(), True),
            StructField("velocity", DoubleType(), True),
            StructField("true_track", DoubleType(), True),
            StructField("vertical_rate", DoubleType(), True),
            StructField("geo_altitude", DoubleType(), True),
            StructField("spi", BooleanType(), True),
            StructField("position_source", IntegerType(), True),
            StructField("category", IntegerType(), True)
        ])
    ), True)
])
 

In [4]:
streaming_df = kafka_json_df.withColumn("values_json", sf.from_json(sf.col("value"), json_schema))
streaming_df = streaming_df.selectExpr("values_json.*")

In [5]:
streaming_df.printSchema()

root
 |-- time: long (nullable = true)
 |-- states: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- icao24: string (nullable = true)
 |    |    |-- callsign: string (nullable = true)
 |    |    |-- origin_country: string (nullable = true)
 |    |    |-- time_position: long (nullable = true)
 |    |    |-- last_contact: long (nullable = true)
 |    |    |-- longitude: double (nullable = true)
 |    |    |-- latitude: double (nullable = true)
 |    |    |-- baro_altitude: double (nullable = true)
 |    |    |-- on_ground: boolean (nullable = true)
 |    |    |-- velocity: double (nullable = true)
 |    |    |-- true_track: double (nullable = true)
 |    |    |-- vertical_rate: double (nullable = true)
 |    |    |-- geo_altitude: double (nullable = true)
 |    |    |-- spi: boolean (nullable = true)
 |    |    |-- position_source: integer (nullable = true)
 |    |    |-- category: integer (nullable = true)



In [6]:
streaming_df = (streaming_df.withColumn("states", sf.explode("states"))
      .withColumn("icao24", sf.col("states.icao24"))
      .withColumn("callsign", sf.col("states.callsign"))
      .withColumn("origin_country", sf.col("states.origin_country"))
      .withColumn("time_position", sf.col("states.time_position"))
      .withColumn("last_contact", sf.col("states.last_contact"))
      .withColumn("longitude", sf.col("states.longitude"))
      .withColumn("latitude", sf.col("states.latitude"))
      .withColumn("baro_altitude", sf.col("states.baro_altitude"))
      .withColumn("on_ground", sf.col("states.on_ground"))
      .withColumn("velocity", sf.col("states.velocity"))
      .withColumn("true_track", sf.col("states.true_track"))
      .withColumn("vertical_rate", sf.col("states.vertical_rate"))
      .withColumn("geo_altitude", sf.col("states.geo_altitude"))
      .withColumn("spi", sf.col("states.spi"))
      .withColumn("position_source", sf.col("states.position_source"))
      .withColumn("true_track", sf.col("states.true_track"))
      .withColumn("category", sf.col("states.category"))
      .withColumn("vertical_rate", sf.col("states.vertical_rate"))
      .drop("states")
)

In [7]:
streaming_df.printSchema()

root
 |-- time: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- origin_country: string (nullable = true)
 |-- time_position: long (nullable = true)
 |-- last_contact: long (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- baro_altitude: double (nullable = true)
 |-- on_ground: boolean (nullable = true)
 |-- velocity: double (nullable = true)
 |-- true_track: double (nullable = true)
 |-- vertical_rate: double (nullable = true)
 |-- geo_altitude: double (nullable = true)
 |-- spi: boolean (nullable = true)
 |-- position_source: integer (nullable = true)
 |-- category: integer (nullable = true)



In [8]:
(streaming_df.writeStream
 .format("parquet")
 .outputMode("append")
 .trigger(processingTime="35 seconds")
 .option("path", "../data/output")
 .option("checkpointLocation", "../data/checkpoint")
 .start()
 .awaitTermination())

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [9]:
df_sd = (spark.read.parquet("../data/output")
)

df_sd.show()

+----------+------+--------+--------------+-------------+------------+---------+--------+-------------+---------+--------+----------+-------------+------------+-----+---------------+--------+
|      time|icao24|callsign|origin_country|time_position|last_contact|longitude|latitude|baro_altitude|on_ground|velocity|true_track|vertical_rate|geo_altitude|  spi|position_source|category|
+----------+------+--------+--------------+-------------+------------+---------+--------+-------------+---------+--------+----------+-------------+------------+-----+---------------+--------+
|1786028234|e48be3|        |        Brazil|   1786028233|  1786028233| -46.6579|-23.6262|         NULL|     true|    3.34|    149.06|         NULL|        NULL|false|              0|       0|
|1786028234|e49f87|PPSGA   |        Brazil|   1786028233|  1786028233| -46.4485|-23.5384|        952.5|    false|   51.96|    270.57|        -3.25|      975.36|false|              0|       0|
|1786028234|e48ad5|TAM3097 |        Braz